In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# Load data
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

# Split & scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("="*60)
print("IRIS CLASSIFICATION - MODEL COMPARISON")
print("="*60)

# Train & compare 4 models
models = {
    'Logistic Regression': (LogisticRegression(max_iter=200, random_state=42), True),
    'Random Forest': (RandomForestClassifier(n_estimators=100, random_state=42), False),
    'SVM': (SVC(kernel='rbf', probability=True, random_state=42), True),
    'XGBoost': (XGBClassifier(n_estimators=100, random_state=42, verbose=0), False)
}

results = {}
model_objs = {}

for name, (model, use_scaled) in models.items():
    X_tr = X_train_scaled if use_scaled else X_train
    X_te = X_test_scaled if use_scaled else X_test
    
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    
    model_objs[name] = model
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, average='weighted'),
        'Recall': recall_score(y_test, y_pred, average='weighted'),
        'F1-Score': f1_score(y_test, y_pred, average='weighted')
    }

results_df = pd.DataFrame(results).T
print("\nModel Comparison:")
print(results_df.round(4))
print("\nHyperparameter Tuning (XGBoost)...")
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1],
    'n_estimators': [50, 100]
}
 
grid = GridSearchCV(XGBClassifier(random_state=42), param_grid, cv=5, n_jobs=-1, verbose=0)
grid.fit(X_train, y_train)
 
best_model = grid.best_estimator_
y_pred_tuned = best_model.predict(X_test)
accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
 
print(f"Best params: {grid.best_params_}")
print(f"Best CV score: {grid.best_score_:.4f}")
print(f"Test accuracy (tuned): {accuracy_tuned:.4f}")
 
# Cross-validation
cv_scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring='accuracy')
print(f"5-Fold CV: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# Feature importance
importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': best_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nFeature Importance:")
print(importance)
import os
os.makedirs('/mnt/user-data/outputs', exist_ok=True)
 
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
 
# Model comparison
results_df[['Accuracy']].plot(kind='bar', ax=axes[0, 0], legend=False, color='steelblue')
axes[0, 0].set_title('Model Accuracy', fontweight='bold')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].set_xticklabels(axes[0, 0].get_xticklabels(), rotation=45, ha='right')
 
# Confusion matrix
cm = confusion_matrix(y_test, y_pred_tuned)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 1], 
            xticklabels=target_names, yticklabels=target_names)
axes[0, 1].set_title('Confusion Matrix (Best Model)', fontweight='bold')
 
# Feature importance
axes[1, 0].barh(importance['Feature'], importance['Importance'], color='forestgreen')
axes[1, 0].set_xlabel('Importance')
axes[1, 0].set_title('Feature Importance', fontweight='bold')
axes[1, 0].invert_yaxis()
 
# All metrics
results_df.plot(kind='bar', ax=axes[1, 1])
axes[1, 1].set_title('All Metrics Comparison', fontweight='bold')
axes[1, 1].set_ylabel('Score')
axes[1, 1].set_xticklabels(axes[1, 1].get_xticklabels(), rotation=45, ha='right')
axes[1, 1].legend(loc='lower right', fontsize=8)
 
plt.tight_layout()
plt.savefig('iris_classification_short.png', dpi=300, bbox_inches='tight')
print("\n✓ Visualization saved: iris_classification_short.png")
 
# Classification report
print("\nClassification Report (Best Model):")
print(classification_report(y_test, y_pred_tuned, target_names=target_names))
 
print("\n" + "="*60)
print("✓ PROJECT COMPLETE!")
print("="*60)